In [1]:
from pyspark.sql import SparkSession
import getpass
username =getpass.getuser()
spark =SparkSession.builder.config('spark.ui.port','0') \
.config('spark.shuffle.useOldFetchProtocol','true') \
.config('spark.sql.warehouse.dir', f'/user/{username}/warehouse') \
.enableHiveSupport() \
.master('yarn') \
.getOrCreate()

In [2]:
from pyspark.sql.functions import current_timestamp,regexp_replace,col,when,length

In [3]:
customers_schema = 'member_id string, emp_title string, emp_length string,home_ownership string,annual_inc float,addr_State string,zip_code string,country string, grade string,sub_grade string,verification_status string,tot_hi_cred_lim float,application_type string,annual_inc_joint float,verification_status_joint string'

In [4]:
customers_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(customers_schema) \
.load(f"/user/{username}/lendingclub/raw/customers_data_csv")

In [5]:
customers_raw_df

member_id,emp_title,emp_length,home_ownership,annual_inc,addr_State,zip_code,country,grade,sub_grade,verification_status,tot_hi_cred_lim,application_type,annual_inc_joint,verification_status_joint
d86a1c7244ace602c...,Admin,3 years,RENT,31000.0,OR,970xx,USA,B,B3,Source Verified,49481.0,Individual,null,null
25b8cd9ff47e50eea...,Teacher,10+ years,MORTGAGE,55983.0,TX,780xx,USA,C,C2,Source Verified,210551.0,Individual,null,null
f9773942305b2dc9c...,Behavioral Asst,10+ years,MORTGAGE,40000.0,VA,201xx,USA,F,F1,Source Verified,272608.0,Individual,null,null
b22f99c198f3738a7...,owner,< 1 year,RENT,20000.0,NC,274xx,USA,C,C3,Source Verified,58521.0,Individual,null,null
4b3b7068c15b0e046...,Supervisor,< 1 year,RENT,45000.0,NY,117xx,USA,C,C5,Source Verified,25500.0,Individual,null,null
0c7a9f8a6d5b209d2...,case manager,1 year,RENT,35000.0,NV,891xx,USA,B,B1,Verified,37588.0,Individual,null,null
f1cc7e6ae9c735148...,null,null,MORTGAGE,90000.0,AZ,851xx,USA,B,B5,Verified,303921.0,Individual,null,null
8f8faa7eeb2a1ee41...,Merchandiser,6 years,RENT,22000.0,NY,145xx,USA,D,D1,Source Verified,17650.0,Individual,null,null
dad3f5ea8788186cb...,executive assistant,1 year,RENT,41000.0,WA,981xx,USA,C,C5,Verified,29975.0,Individual,null,null
a20867c0985126202...,Industrial Tech,10+ years,MORTGAGE,75000.0,KS,676xx,USA,B,B4,Not Verified,68593.0,Individual,null,null


In [6]:
customers_raw_df.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: float (nullable = true)
 |-- addr_State: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- tot_hi_cred_lim: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- annual_inc_joint: float (nullable = true)
 |-- verification_status_joint: string (nullable = true)



In [7]:
#withColumnsRenamed can be used since spark 3.4
customers_renamed = customers_raw_df.withColumnRenamed('annual_inc', 'annual_income') \
.withColumnRenamed('addr_state', 'address_state') \
.withColumnRenamed('zip_cod', 'address_zipcode')\
.withColumnRenamed('country', 'address_country') \
.withColumnRenamed('tot_hi_cred_lim', 'total_high_credit_limit')\
.withColumnRenamed('annual_inc_joint', 'join_annual_income')

In [8]:
# insert a new column named as ingestion date (current time)
customers_df_ingestion = customers_renamed.withColumn('ingestion_date',current_timestamp())

In [9]:
customers_df_ingestion

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,zip_code,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingestion_date
d86a1c7244ace602c...,Admin,3 years,RENT,31000.0,OR,970xx,USA,B,B3,Source Verified,49481.0,Individual,null,null,2026-03-28 02:17:...
25b8cd9ff47e50eea...,Teacher,10+ years,MORTGAGE,55983.0,TX,780xx,USA,C,C2,Source Verified,210551.0,Individual,null,null,2026-03-28 02:17:...
f9773942305b2dc9c...,Behavioral Asst,10+ years,MORTGAGE,40000.0,VA,201xx,USA,F,F1,Source Verified,272608.0,Individual,null,null,2026-03-28 02:17:...
b22f99c198f3738a7...,owner,< 1 year,RENT,20000.0,NC,274xx,USA,C,C3,Source Verified,58521.0,Individual,null,null,2026-03-28 02:17:...
4b3b7068c15b0e046...,Supervisor,< 1 year,RENT,45000.0,NY,117xx,USA,C,C5,Source Verified,25500.0,Individual,null,null,2026-03-28 02:17:...
0c7a9f8a6d5b209d2...,case manager,1 year,RENT,35000.0,NV,891xx,USA,B,B1,Verified,37588.0,Individual,null,null,2026-03-28 02:17:...
f1cc7e6ae9c735148...,null,null,MORTGAGE,90000.0,AZ,851xx,USA,B,B5,Verified,303921.0,Individual,null,null,2026-03-28 02:17:...
8f8faa7eeb2a1ee41...,Merchandiser,6 years,RENT,22000.0,NY,145xx,USA,D,D1,Source Verified,17650.0,Individual,null,null,2026-03-28 02:17:...
dad3f5ea8788186cb...,executive assistant,1 year,RENT,41000.0,WA,981xx,USA,C,C5,Verified,29975.0,Individual,null,null,2026-03-28 02:17:...
a20867c0985126202...,Industrial Tech,10+ years,MORTGAGE,75000.0,KS,676xx,USA,B,B4,Not Verified,68593.0,Individual,null,null,2026-03-28 02:17:...


In [10]:
#4. remove complete duplicate rows
de_duplicate = customers_df_ingestion.distinct()

In [11]:
de_duplicate.count()

2260638

In [12]:
de_duplicate.createOrReplaceTempView("customers")

In [13]:
spark.sql("select count(*) from customers where annual_income is null")

count(1)
5


In [14]:
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")

In [15]:
customers_income_filtered

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,zip_code,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingestion_date
5ac689ee6199daa95...,Operator,3 years,MORTGAGE,35000.0,VA,241xx,USA,B,B3,Source Verified,169866.0,Individual,null,null,2026-03-28 02:18:...
29970a3d0b8482208...,null,null,RENT,120000.0,NY,112xx,USA,A,A4,Source Verified,28900.0,Individual,null,null,2026-03-28 02:18:...
f0cf3060eddd9d7cc...,Logistics Coordin...,7 years,RENT,66000.0,MA,017xx,USA,A,A3,Source Verified,115175.0,Individual,null,null,2026-03-28 02:18:...
7e7a1d5a026ba7b25...,Operations,9 years,OWN,55000.0,AL,352xx,USA,A,A3,Not Verified,190723.0,Individual,null,null,2026-03-28 02:18:...
20277685236669da5...,OTR DRIVER,10+ years,MORTGAGE,70450.0,FL,328xx,USA,A,A5,Not Verified,183848.0,Individual,null,null,2026-03-28 02:18:...
aea502dac22436fc5...,null,null,RENT,34000.0,NY,100xx,USA,B,B5,Source Verified,11300.0,Individual,null,null,2026-03-28 02:18:...
e497e689b640e36bd...,President,10+ years,MORTGAGE,320000.0,VA,220xx,USA,B,B3,Verified,186024.0,Individual,null,null,2026-03-28 02:18:...
950f47b9ca9e7b025...,Machinist,10+ years,MORTGAGE,97000.0,MI,480xx,USA,D,D2,Source Verified,208013.0,Individual,null,null,2026-03-28 02:18:...
cd20ff2770fce6dbc...,Millwright/welder,10+ years,MORTGAGE,69000.0,AL,350xx,USA,C,C5,Source Verified,275134.0,Individual,null,null,2026-03-28 02:18:...
64f78473ca1aad803...,Lead team technician,10+ years,RENT,54000.0,VT,052xx,USA,D,D2,Verified,48046.0,Individual,null,null,2026-03-28 02:18:...


In [16]:
customers_income_filtered.createOrReplaceTempView("customers")

In [17]:
spark.sql("select distinct(emp_length) from customers").show()

+----------+
|emp_length|
+----------+
|   9 years|
|   5 years|
|      null|
|    1 year|
|   2 years|
|   7 years|
|   8 years|
|   4 years|
|   6 years|
|   3 years|
| 10+ years|
|  < 1 year|
+----------+



In [18]:
customers_emplength_cleaned = customers_income_filtered.withColumn("emp_length",regexp_replace(col("emp_length"),"(\D)",""))

In [19]:
customers_emplength_cleaned

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,zip_code,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingestion_date
3f8c8b1af2c3851d0...,null,null,OWN,57000.0,PA,190xx,USA,C,C3,Source Verified,6400.0,Individual,null,null,2026-03-28 02:18:...
bf0a0c3e81115ba2d...,LPN,1,MORTGAGE,30000.0,NY,121xx,USA,E,E1,Verified,34199.0,Individual,null,null,2026-03-28 02:18:...
4cf15235f3085c3b6...,Dispatcher,8,RENT,35000.0,CA,925xx,USA,E,E3,Verified,49133.0,Individual,null,null,2026-03-28 02:18:...
8e9072f04707750c7...,Instructional Des...,10,MORTGAGE,81900.0,AZ,852xx,USA,B,B5,Not Verified,21600.0,Individual,null,null,2026-03-28 02:18:...
f9d31b4819bbcae66...,senior equal empl...,10,MORTGAGE,106000.0,TX,773xx,USA,B,B2,Not Verified,231787.0,Individual,null,null,2026-03-28 02:18:...
2c0bf17700813f87e...,Union carpenter,2,MORTGAGE,85000.0,WI,539xx,USA,B,B5,Source Verified,138837.0,Individual,null,null,2026-03-28 02:18:...
a177be0e06dd3a307...,processor2,10,MORTGAGE,30000.0,TN,381xx,USA,B,B4,Source Verified,93243.0,Individual,null,null,2026-03-28 02:18:...
3975c27c1a20feebd...,Network Administr...,4,MORTGAGE,50000.0,FL,337xx,USA,A,A3,Not Verified,254059.0,Individual,null,null,2026-03-28 02:18:...
a27ba7ae904788fbc...,Field Technology ...,2,RENT,34179.0,MD,207xx,USA,C,C5,Verified,14146.0,Individual,null,null,2026-03-28 02:18:...
aa9540fa495133094...,Administrative Su...,7,OWN,114000.0,MO,648xx,USA,A,A3,Not Verified,64570.0,Individual,null,null,2026-03-28 02:18:...


In [20]:
customers_emplength_cleaned.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_income: float (nullable = true)
 |-- address_state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- total_high_credit_limit: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- join_annual_income: float (nullable = true)
 |-- verification_status_joint: string (nullable = true)
 |-- ingestion_date: timestamp (nullable = false)



In [21]:
customers_emplength_cast = customers_emplength_cleaned.withColumn("emp_length",customers_emplength_cleaned.emp_length.cast('int'))

In [22]:
customers_emplength_cast.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: integer (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_income: float (nullable = true)
 |-- address_state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- total_high_credit_limit: float (nullable = true)
 |-- application_type: string (nullable = true)
 |-- join_annual_income: float (nullable = true)
 |-- verification_status_joint: string (nullable = true)
 |-- ingestion_date: timestamp (nullable = false)



In [23]:
customers_emplength_cast.filter("emp_length is null").count()

146903

In [24]:
customers_emplength_cast.createOrReplaceTempView("customers")

In [25]:
avg_emp_length = spark.sql("select floor(avg(emp_length)) as avg_emp_length from customers").collect()

In [26]:
avg_emp_length

[Row(avg_emp_length=6)]

In [27]:
avg_emp_duration = avg_emp_length[0][0]

In [28]:
avg_emp_duration

6

In [29]:
customers_emplength_replaced = customers_emplength_cast.na.fill(avg_emp_duration,subset=['emp_length'])

In [30]:
customers_emplength_replaced.filter("emp_length is null").count()

0

In [31]:
customers_emplength_replaced.createOrReplaceTempView("customers")

In [32]:
spark.sql("select distinct(address_state) from customers")

address_state
Helping Kenya's D...
223xx
175 (total projec...
AZ
SC
"so Plan """"C"""" is ..."
I am 56 yrs. old ...
financially I mad...
but no one will l...
LA


In [33]:
spark.sql("select count(address_state) from customers where length(address_state)>2")

count(address_state)
254


In [36]:
customers_state_cleaned = customers_emplength_replaced.withColumn(
    "address_state",
    when(length(col("address_state"))>2,"NA").otherwise(col("address_state"))
)

In [37]:
customers_state_cleaned

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,zip_code,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingestion_date
9dc4c90c4259f6c1b...,Area Trainer,5,RENT,60000.0,OH,432xx,USA,E,E4,Source Verified,150615.0,Individual,null,null,2026-03-28 02:19:...
d6952722b5de341ac...,Industrial Design...,4,RENT,87000.0,FL,327xx,USA,A,A5,Not Verified,37671.0,Individual,null,null,2026-03-28 02:19:...
d00c0e6a0f2079ccf...,Fab tech.,10,MORTGAGE,45000.0,CO,800xx,USA,C,C3,Verified,211532.0,Individual,null,null,2026-03-28 02:19:...
13fecd601f86bc19c...,heavy equipment o...,2,RENT,50000.0,ND,585xx,USA,B,B1,Source Verified,55093.0,Individual,null,null,2026-03-28 02:19:...
0e2da8f5e8f3c51b2...,owner/consultant ...,6,MORTGAGE,108000.0,TN,372xx,USA,A,A3,Not Verified,356023.0,Individual,null,null,2026-03-28 02:19:...
99a0f5a9dc7ec36ca...,Driver,10,OWN,75000.0,NY,125xx,USA,B,B3,Not Verified,332641.0,Individual,null,null,2026-03-28 02:19:...
b0bf169efefdd7bef...,Professor,10,MORTGAGE,117159.78,TX,773xx,USA,C,C4,Verified,375560.0,Individual,null,null,2026-03-28 02:19:...
62184ea5c9868ec4e...,Lt. Colonel,10,RENT,127000.0,SC,292xx,USA,A,A5,Not Verified,382640.0,Individual,null,null,2026-03-28 02:19:...
1e8c4a36a14912462...,Refrigeration mec...,10,MORTGAGE,79000.0,NE,685xx,USA,C,C2,Verified,44300.0,Individual,null,null,2026-03-28 02:19:...
69bf5908d2fc075b3...,Owner,10,OWN,70000.0,MS,395xx,USA,B,B5,Not Verified,152078.0,Individual,null,null,2026-03-28 02:19:...


In [38]:
customers_state_cleaned.select("address_state").distinct()

address_state
SC
AZ
LA
MN
NJ
DC
OR
NA
VA
null


In [39]:
customers_state_cleaned.write \
.format("parquet") \
.mode("overwrite") \
.option("path",(f"/user/{username}/lendingclub/cleaned/customers_parquet")) \
.save()

In [40]:
customers_state_cleaned.write \
.format("parquet") \
.mode("overwrite") \
.option("path",(f"/user/{username}/lendingclub/cleaned/customers_parquet")) \
.save()